<a href="https://colab.research.google.com/github/suhailahmed10/master/blob/main/F1_Winner_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [ ]:
#  Pre-fill 2026 Australian GP drivers and constructors
drivers = [
"Max Verstappen","Isack Hadjar",
"Lando Norris","Oscar Piastri",
"Charles Leclerc","Lewis Hamilton",
"George Russell","Kimi Antonelli",
"Fernando Alonso","Lance Stroll",
"Esteban Ocon","Oliver Bearman",
"Nico Hulkenberg","Gabriel Bortoleto",
"Alex Albon","Carlos Sainz",
"Liam Lawson","Arvid Lindblad",
"Pierre Gasly","Franco Colapinto",
"Valtteri Bottas","Sergio Perez"
]
constructors = [
"Red Bull","Red Bull",
"McLaren","McLaren",
"Ferrari","Ferrari",
"Mercedes","Mercedes",
"Aston Martin","Aston Martin",
"Haas","Haas",
"Audi","Audi",
"Williams","Williams",
"Racing Bulls","Racing Bulls",
"Alpine","Alpine",
"Cadillac","Cadillac"
]

# nput Qualifying Times (seconds)
# Replace these values with real 2026 qualifying times when available
qual_times =[ 00.000,  #Max
79.303, #Isack Hadjar
79.475, #Lando Norris
79.380,  #Oscar Piastri
79.327, #Charles Leclerc
79.478, #Lewis Hamilton
78.518, #George Russell
78.811, #Kimi Antonelli
81.969, #Fernando Alonso
00.000, #Lance Stroll
80.491, #Esteban Ocon
80.303, #Oliver Bearman
80.303, #Nico Hulkenberg
80.211, #Gabriel Bortoleto
80.941, #Alex Albon
00.000, #Carlos Sainz
79.994, #Liam Lawson
81.247, #Arvid Lindblad
80.501, #Pierre Gasly
81.270, #Franco Colapinto
83.244, #Valtteri Bottas
82.605 #Sergio Perez
]

# Create DataFrame
df = pd.DataFrame({
    "Driver": drivers,
    "Constructor": constructors,
    "QualTime": qual_times
})

# Calculate Gap to Pole and Qualifying Rank
pole_time = df["QualTime"].min()
df["GapToPole"] = df["QualTime"] - pole_time
df["QualRank"] = df["QualTime"].rank().astype(int)

# Feature Engineering for ML
np.random.seed(42)
df["DriverSkill"] = np.random.normal(0, 1, len(df))  # simulated driver skill
df["ConstructorStrength"] = np.random.normal(0, 1, len(df))  # simulated car strength
df["Grid"] = df["QualRank"] + np.random.choice([-1,0,1], size=len(df))
df["Grid"] = df["Grid"].clip(1, len(df))

# Target Race Position (simulated)
df["RacePos"] = df["QualRank"] + np.random.normal(0, 2, len(df)) - df["DriverSkill"] + df["ConstructorStrength"]
df["RacePos"] = df["RacePos"].round().clip(1, len(df)).astype(int)

df.head(10)

,Driver,Constructor,QualTime,GapToPole,QualRank,DriverSkill,ConstructorStrength,Grid,RacePos
0,Max Verstappen,Red Bull,0.000,0.000,2,0.496714,0.067528,1,1
1,Isack Hadjar,Red Bull,79.303,79.303,6,-0.138264,-1.424748,6,4
2,Lando Norris,McLaren,79.475,79.475,9,0.647689,-0.544383,10,8
3,Oscar Piastri,McLaren,79.380,79.380,8,1.523030,0.110923,7,9
4,Charles Leclerc,Ferrari,79.327,79.327,7,-0.234153,-1.150994,7,5
5,Lewis Hamilton,Ferrari,79.478,79.478,10,-0.234137,0.375698,9,10
6,George Russell,Mercedes,78.518,78.518,4,1.579213,-0.600639,3,1
7,Kimi Antonelli,Mercedes,78.811,78.811,5,0.767435,-0.291694,4,2
8,Fernando Alonso,Aston Martin,81.969,81.969,20,-0.469474,-0.601707,19,21
9,Lance Stroll,Aston Martin,0.000,0.000,2,0.542560,1.852278,3,6


In [ ]:
# Replace 0.000 (DNS) with max + buffer
dns_drivers = df[df["QualTime"] == 0.000]["Driver"].tolist()
df.loc[df["QualTime"] == 0.000, "QualTime"] = df["QualTime"].replace(0, np.nan)
df["QualTime"].fillna(df["QualTime"].max() + 5, inplace=True)

# Recalculate GapToPole and QualRank
pole_time = df["QualTime"].min()
df["GapToPole"] = df["QualTime"] - pole_time
df["QualRank"] = df["QualTime"].rank().astype(int)

# Recalculate Grid with DNS drivers at back
df["Grid"] = df["QualRank"] + np.random.choice([-1,0,1], size=len(df))
df["Grid"] = df["Grid"].clip(1, len(df))
df.loc[df["Driver"].isin(dns_drivers), "Grid"] = len(df)

In [ ]:
np.random.seed(42)
df["DriverSkill"] = np.random.normal(0, 1, len(df))
df["ConstructorStrength"] = np.random.normal(0, 1, len(df))

# Features for ML
features = ["QualTime","GapToPole","QualRank","Grid","DriverSkill","ConstructorStrength","Constructor"]
X = pd.get_dummies(df[features], drop_first=True)
y = df["RacePos"].astype(int)  # integer positions

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train classifier
clf = RandomForestClassifier(n_estimators=1000, max_depth=12, random_state=42)
clf.fit(X, y)

# Predict probabilities
y_proba = clf.predict_proba(X)

# Compute expected position for each driver
df["PredictedRacePos"] = np.dot(y_proba, clf.classes_).round(2)

# Sort for top 3
df_pred = df.sort_values("PredictedRacePos").reset_index(drop=True)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# True race positions
y_true = df["RacePos"]

# Predicted race positions
y_pred = df["PredictedRacePos"]

# Metrics
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.2f}")

Mean Absolute Error (MAE): 1.19
Mean Squared Error (MSE): 2.55
R² Score: 0.95


In [ ]:
from IPython.display import display, HTML

# Top 3 drivers
top3 = df_pred.head(3).reset_index(drop=True)

# Assign medal icons
medals = ["🥇","🥈","🥉"]  # gold, silver, bronze
top3["Medal"] = medals

# Select columns to display
top3_display = top3[["Medal","Driver","Constructor","PredictedRacePos","QualRank","QualTime"]]

# Display as HTML table
def render_medal_table(df):
    styles = [
        dict(selector="th", props=[("font-size", "14pt"), ("text-align", "center")]),
        dict(selector="td", props=[("text-align", "center"), ("font-size", "12pt")]),
        dict(selector="caption", props=[("caption-side", "top"), ("font-size", "16pt")])
    ]
    return df.style.set_table_styles(styles).set_caption("Predicted Podium - 2026 Australian GP")

render_medal_table(top3_display)

,Medal,Driver,Constructor,PredictedRacePos,QualRank,QualTime
0,🥇,George Russell,Mercedes,2.170000,1,78.518000
1,🥈,Kimi Antonelli,Mercedes,2.630000,2,78.811000
2,🥉,Max Verstappen,Red Bull,5.040000,21,93.244000


In [ ]:
# Clean, presentable table for team
df_pred[["PredictedRacePos","Driver","Constructor","QualRank","QualTime","GapToPole","Grid"]].style.background_gradient(cmap='viridis').format({
    "PredictedRacePos": "{:.2f}",
    "QualTime": "{:.3f}",
    "GapToPole": "{:.3f}"
})

,PredictedRacePos,Driver,Constructor,QualRank,QualTime,GapToPole,Grid
0,2.17,George Russell,Mercedes,1,78.518,0.000,1
1,2.63,Kimi Antonelli,Mercedes,2,78.811,0.293,1
2,5.04,Max Verstappen,Red Bull,21,93.244,14.726,21
3,5.13,Isack Hadjar,Red Bull,3,79.303,0.785,4
4,5.78,Charles Leclerc,Ferrari,4,79.327,0.809,3
5,6.46,Carlos Sainz,Williams,22,98.244,19.726,22
6,8.07,Lando Norris,McLaren,6,79.475,0.957,6
7,8.37,Oscar Piastri,McLaren,5,79.380,0.862,6
8,9.55,Lewis Hamilton,Ferrari,7,79.478,0.960,8
9,9.82,Lance Stroll,Aston Martin,20,88.244,9.726,19


In [ ]:
# Save predicted race results
df_pred.to_csv("predicted_race_results.csv", index=False)